# Qwen3.5-4B: OpenAI-compatible LLM server on Colab

This notebook serves **`Qwen/Qwen3.5-4B`** on a Colab GPU and exposes it as a secured,
OpenAI-compatible endpoint through a public HTTPS tunnel, for the `drive-failure-agents` backend.

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Optional Colab **Secrets** (key icon on the left):
   - `HF_TOKEN`: faster model downloads;
   - `NGROK_TOKEN`: only if you choose the ngrok tunnel;
   - `LLM_API_KEY`: keeps the same key across sessions (otherwise a random one is generated).

   Allow notebook access when Colab asks.
3. **Runtime → Run all.** The first run takes about 10–25 minutes (installs and model download).
4. Copy the `LLM_*` block printed in step 9 into `server/.env`, or paste it on the web app's
   **Settings** page.

**How it works**: `tunnel → proxy (port PORT) → backend (port PORT+1)`.
- The **proxy** checks the API key, turns thinking off by default, removes `<think>` text, and
  returns readable errors.
- The **backend** is **vLLM** if it passes a strict validation: correct answers, JSON,
  thinking mode, streaming and a long generation. Otherwise vLLM is stopped and a
  **transformers** server is used. If neither passes, the notebook stops and prints the reason
  and the logs.

**Important**
- The tunnel URL **changes every session**: update `LLM_BASE_URL` each time.
- Free Colab sessions **disconnect** after idle time or a maximum runtime. The monitor cell (11)
  keeps the session active while the tab stays open.
- Requests without the API key are rejected (HTTP 401).
- If anything fails, run the **Troubleshooting** cell at the very end and read its output.

## 1 · Instructions
See the cell above. Every following cell is part of **Run all**, except Shutdown and Troubleshooting, which never run on their own.

## 2 · GPU check
Fails fast if no CUDA GPU is attached.

In [ ]:
import subprocess, sys
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Use Runtime → Change runtime type → T4 GPU, then Run all again.")
_p = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_MEM_GB, GPU_CC = _p.name, _p.total_memory / 1024**3, (_p.major, _p.minor)
print(f"GPU: {GPU_NAME} | {GPU_MEM_GB:.1f} GB | compute capability {GPU_CC[0]}.{GPU_CC[1]}")
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | CUDA {torch.version.cuda}")
if GPU_MEM_GB < 14:
    print("WARNING: less than 14 GB GPU memory; consider lowering MAX_MODEL_LEN.")

## 3 · Configuration
- `BACKEND`:
  - `auto`: try vLLM, fall back to transformers;
  - `vllm`: vLLM only;
  - `transformers`: skip vLLM. The most robust choice on a T4, but slower.
- `ENABLE_THINKING=False` sends `chat_template_kwargs={"enable_thinking": false}` by default;
  clients may override it per request.
- T4 has no bf16 support, so the model runs in **float16**. `VLLM_ENFORCE_EAGER` avoids CUDA-graph
  capture, which is a common start-up failure on T4.

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"  #@param {type:"string"}
MAX_MODEL_LEN = 8192  #@param {type:"integer"}
ENABLE_THINKING = False  #@param {type:"boolean"}
TUNNEL = "cloudflared"  #@param ["cloudflared", "ngrok"]
PORT = 8000  #@param {type:"integer"}
BACKEND = "auto"  #@param ["auto", "vllm", "transformers"]
STARTUP_TIMEOUT_S = 1500  #@param {type:"integer"}
GPU_MEMORY_UTILIZATION = 0.85  #@param {type:"number"}
VLLM_ENFORCE_EAGER = True  #@param {type:"boolean"}

import os, pathlib
BACKEND_PORT = PORT + 1
WORKDIR = pathlib.Path("/content/llm_server"); WORKDIR.mkdir(parents=True, exist_ok=True)
LOGS = {name: WORKDIR / f"{name}.log" for name in ("install", "vllm", "fallback", "proxy", "tunnel")}
VLLM_ENV = pathlib.Path("/content/venvs/vllm")  # isolated: vLLM never touches Colab's packages

# Pinned versions (checked October 2026).
VERSIONS = {
    "vllm": "0.32.0",
    "vllm_python": "3.12",
    "uv": "0.13.5",
    "transformers": "5.20.2",
    "fastapi": "0.162.0",
    "uvicorn": "0.54.0",
    "httpx": "0.28.1",
    "openai": "3.37.0",
    "pydantic": "2.14.1",
    "pyngrok": "9.0.3",
    "cloudflared": "2026.10.3",
}
STATE = {"backend": None, "vllm_error": None, "fallback_error": None}
print({k: v for k, v in globals().items() if k.isupper() and k not in ("LOGS", "STATE")})

## 4 · Secrets
Reads `HF_TOKEN` (optional), `NGROK_TOKEN` (ngrok only) and `LLM_API_KEY` (optional) from Colab
Secrets. If no `LLM_API_KEY` secret exists, a random key is generated and printed once.

In [ ]:
import secrets as _secrets
try:
    from google.colab import userdata
except ImportError:  # running outside Colab
    userdata = None

def colab_secret(name):
    if userdata is None:
        return os.environ.get(name)
    try:
        return userdata.get(name)
    except Exception:
        return None

HF_TOKEN = colab_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
NGROK_TOKEN = colab_secret("NGROK_TOKEN") if TUNNEL == "ngrok" else None
if TUNNEL == "ngrok" and not NGROK_TOKEN:
    raise RuntimeError("TUNNEL='ngrok' needs a NGROK_TOKEN Colab secret (or switch to cloudflared).")

LLM_API_KEY = colab_secret("LLM_API_KEY")
if not LLM_API_KEY:
    LLM_API_KEY = "sk-" + _secrets.token_urlsafe(32)
    print("Generated a new LLM_API_KEY (shown once here and in the connection block):")
    print(LLM_API_KEY)
else:
    print("Using LLM_API_KEY from Colab Secrets.")
BACKEND_KEY = _secrets.token_urlsafe(24)  # internal key between proxy and backend
print("HF_TOKEN:", "set" if HF_TOKEN else "not set (anonymous download)")

## 5 · Install (pinned)
- Proxy, validator and transformers fallback go into Colab's own Python.
- **vLLM** goes into a separate virtual environment (`/content/venvs/vllm`, Python 3.12, created
  with `uv`). Its own torch and transformers versions cannot break the fallback.
- Installer output is written to `install.log`; failures print its tail.

In [ ]:
import shutil, subprocess, sys, time

def run(cmd, log=LOGS["install"], timeout=None):
    """Run a command, append its output to a log; returns True on success."""
    print("$", " ".join(map(str, cmd)))
    with open(log, "a") as fh:
        fh.write(f"\n$ {' '.join(map(str, cmd))}\n"); fh.flush()
        try:
            return subprocess.run(list(map(str, cmd)), stdout=fh, stderr=subprocess.STDOUT,
                                  timeout=timeout).returncode == 0
        except subprocess.TimeoutExpired:
            fh.write("TIMEOUT\n")
            return False

def log_tail(path, n=40):
    try:
        return "\n".join(pathlib.Path(path).read_text(errors="replace").splitlines()[-n:])
    except FileNotFoundError:
        return "(no log)"

V = VERSIONS
base = [f"fastapi=={V['fastapi']}", f"uvicorn=={V['uvicorn']}", f"httpx=={V['httpx']}",
        f"openai=={V['openai']}", f"pydantic=={V['pydantic']}", f"transformers=={V['transformers']}"]
if TUNNEL == "ngrok":
    base.append(f"pyngrok=={V['pyngrok']}")
if not run([sys.executable, "-m", "pip", "install", "-q", *base]):
    print(log_tail(LOGS["install"]))
    raise RuntimeError("base install failed (see output above)")

VLLM_INSTALLED = False
if BACKEND in ("auto", "vllm"):
    t0 = time.time()
    VLLM_INSTALLED = (
        run([sys.executable, "-m", "pip", "install", "-q", f"uv=={V['uv']}"])
        and (shutil.rmtree(VLLM_ENV, ignore_errors=True) or True)
        and run(["uv", "venv", VLLM_ENV, "--python", V["vllm_python"], "--seed"])
        and run(["uv", "pip", "install", "--python", VLLM_ENV / "bin" / "python",
                 f"vllm=={V['vllm']}", "--torch-backend=auto"], timeout=2400)
        and (VLLM_ENV / "bin" / "vllm").exists())
    print(f"vLLM install {'OK' if VLLM_INSTALLED else 'FAILED'} in {time.time() - t0:.0f}s")
    if not VLLM_INSTALLED:
        STATE["vllm_error"] = "install failed"
        print(log_tail(LOGS["install"], 25))
        if BACKEND == "vllm":
            raise RuntimeError("vLLM install failed and BACKEND='vllm' was forced.")
print("install step done")

## 6 · Serve the model (OpenAI-compatible)
- The next three cells write the proxy, the fallback server and the validator.
- The cell after them starts a backend and validates it with real requests. A backend is only
  used if every check passes:
  - **plain**: the answer to *17 + 25* contains 42;
  - **json**: a JSON-only answer parses;
  - **thinking**: thinking mode returns an answer;
  - **stream**: streaming works;
  - **long**: a 400-token generation completes.
- vLLM runs with `--dtype float16`, `--max-model-len` and `--language-model-only` (text only),
  with its API key and logs in `vllm.log`. If it fails to start or fails validation, it is
  stopped, the GPU is freed, and the transformers fallback is validated the same way.

In [ ]:
%%writefile /content/llm_server/proxy.py
"""Public-facing OpenAI-compatible proxy.

Sits between the tunnel and the model backend (vLLM or the transformers
fallback) and guarantees, whatever the backend:
  * Bearer-token auth on every route (requests without LLM_API_KEY get 401);
  * thinking control: injects chat_template_kwargs.enable_thinking (default from
    ENABLE_THINKING) when the client does not set it;
  * server-side removal of <think>...</think> reasoning (streaming and not), and
    of any separate reasoning_content field, so clients get clean answers;
  * readable errors: a backend failure is returned with the backend's status and
    message, and a proxy failure as JSON naming the exception (never a bare 500);
  * request / error counters for the notebook's monitor cell (GET /stats).

Configuration comes from environment variables:
  LLM_API_KEY, BACKEND_URL, BACKEND_KEY, ENABLE_THINKING, MODEL_ID
"""

from __future__ import annotations

import json
import os
import secrets
import time
import traceback

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

API_KEY = os.environ["LLM_API_KEY"]
BACKEND_URL = os.environ.get("BACKEND_URL", "http://127.0.0.1:8001").rstrip("/")
BACKEND_KEY = os.environ.get("BACKEND_KEY", "")
DEFAULT_THINKING = os.environ.get("ENABLE_THINKING", "false").lower() == "true"
OPEN, CLOSE = "<think>", "</think>"

STATS = {
    "started_at": time.time(),
    "requests": 0,
    "errors": 0,
    "last_error": None,
    "last_latency_s": None,
    "completion_tokens": 0,
}

app = FastAPI(title="LLM proxy")
client = httpx.AsyncClient(timeout=httpx.Timeout(900.0, connect=10.0))


class ThinkFilter:
    """Incrementally removes reasoning from generated text.

    ``starts_inside`` handles chat templates that pre-fill the opening <think>
    tag in the prompt (the model output then begins inside the reasoning and
    only the closing tag appears).
    """

    def __init__(self, starts_inside: bool = False) -> None:
        self.inside = starts_inside
        self.prefilled = starts_inside  # still inside a reasoning block opened by the prompt
        self.held = ""  # text withheld while possibly inside a prefilled block
        self.buf = ""
        self.emitted_any = False

    def feed(self, text: str) -> str:
        self.buf += text
        out = []
        while True:
            tag = CLOSE if self.inside else OPEN
            idx = self.buf.find(tag)
            if idx >= 0:
                if not self.inside:
                    out.append(self.buf[:idx])
                self.buf = self.buf[idx + len(tag):]
                self.inside = not self.inside
                self.prefilled = False
                self.held = ""
                continue
            # keep a possible partial tag at the end of the buffer
            keep = 0
            for k in range(1, len(tag)):
                if self.buf.endswith(tag[:k]):
                    keep = k
            emit, self.buf = self.buf[: len(self.buf) - keep], self.buf[len(self.buf) - keep:]
            if not self.inside:
                out.append(emit)
            elif self.prefilled:
                self.held += emit
            text_out = "".join(out)
            if not self.emitted_any:
                text_out = text_out.lstrip()
            self.emitted_any = self.emitted_any or bool(text_out)
            return text_out

    def flush(self) -> str:
        rest, self.buf = self.buf, ""
        if self.prefilled:
            # the closing tag never came: there was no reasoning, it was all answer
            return (self.held + rest).strip()
        return "" if self.inside else rest


def strip_think(text: str | None, starts_inside: bool = False) -> str | None:
    """Remove reasoning from a complete response."""
    if not text:
        return text
    head, sep, tail = text.partition(CLOSE)
    if sep and OPEN not in head:
        # orphan closing tag: the opening tag was part of the prompt
        text, starts_inside = tail, False
    f = ThinkFilter(starts_inside=starts_inside)
    return (f.feed(text) + f.flush()).strip()


def _error(status: int, message: str, kind: str = "proxy_error") -> JSONResponse:
    return JSONResponse({"error": {"message": message, "type": kind}}, status_code=status)


def _authorized(request: Request) -> bool:
    header = request.headers.get("authorization", "")
    token = header[7:] if header.lower().startswith("bearer ") else ""
    return bool(token) and secrets.compare_digest(token, API_KEY)


@app.middleware("http")
async def auth_and_errors(request: Request, call_next):
    if not _authorized(request):
        return _error(401, "invalid or missing API key", "auth")
    try:
        return await call_next(request)
    except Exception as exc:  # never answer with an opaque 500
        _record_error(f"{request.url.path}: {exc!r}")
        tb = traceback.format_exc(limit=4)
        print(tb, flush=True)
        return _error(500, f"proxy failed: {type(exc).__name__}: {exc}")


def _backend_headers() -> dict[str, str]:
    return {"Authorization": f"Bearer {BACKEND_KEY}"} if BACKEND_KEY else {}


def _record_error(msg: str) -> None:
    STATS["errors"] += 1
    STATS["last_error"] = f"{time.strftime('%H:%M:%S')} {msg[:500]}"
    print(f"ERROR {STATS['last_error']}", flush=True)


def _safe_json(r: httpx.Response):
    try:
        return r.json()
    except ValueError:
        return {"error": {"message": r.text[:2000] or f"HTTP {r.status_code}",
                          "type": "backend_error"}}


@app.get("/stats")
async def stats():
    return STATS


@app.get("/v1/models")
async def models():
    try:
        r = await client.get(f"{BACKEND_URL}/v1/models", headers=_backend_headers())
    except httpx.HTTPError as exc:
        _record_error(f"/v1/models: {exc!r}")
        return _error(502, f"backend unavailable: {exc!r}", "backend_unavailable")
    return JSONResponse(_safe_json(r), status_code=r.status_code)


def _clean_message(msg: dict, thinking: bool) -> None:
    reasoning = msg.pop("reasoning_content", None)
    msg.pop("reasoning", None)
    content = msg.get("content")
    if isinstance(content, str):
        msg["content"] = strip_think(content)
    if (not msg.get("content")) and reasoning and not thinking:
        # some reasoning parsers put the whole answer in reasoning_content when
        # thinking is disabled; that text is the answer.
        msg["content"] = strip_think(reasoning)


@app.post("/v1/chat/completions")
async def chat(request: Request):
    try:
        body = await request.json()
    except ValueError:
        return _error(400, "request body is not valid JSON", "invalid_request")
    kwargs = body.get("chat_template_kwargs") or {}
    kwargs.setdefault("enable_thinking", DEFAULT_THINKING)
    body["chat_template_kwargs"] = kwargs
    thinking = bool(kwargs["enable_thinking"])
    STATS["requests"] += 1
    t0 = time.time()
    url = f"{BACKEND_URL}/v1/chat/completions"

    if not body.get("stream"):
        try:
            r = await client.post(url, json=body, headers=_backend_headers())
        except httpx.HTTPError as exc:
            _record_error(f"chat: {exc!r}")
            return _error(502, f"backend unavailable: {exc!r}", "backend_unavailable")
        if r.status_code != 200:
            _record_error(f"chat: backend HTTP {r.status_code}: {r.text[:300]}")
            return JSONResponse(_safe_json(r), status_code=r.status_code)
        data = _safe_json(r)
        for choice in data.get("choices", []):
            _clean_message(choice.get("message") or {}, thinking)
        STATS["last_latency_s"] = round(time.time() - t0, 3)
        STATS["completion_tokens"] += (data.get("usage") or {}).get("completion_tokens", 0) or 0
        return JSONResponse(data)

    async def stream():
        filters: dict[int, ThinkFilter] = {}
        try:
            async with client.stream("POST", url, json=body, headers=_backend_headers()) as r:
                if r.status_code != 200:
                    text = (await r.aread()).decode(errors="replace")
                    _record_error(f"stream: backend HTTP {r.status_code}: {text[:300]}")
                    yield f"data: {json.dumps({'error': {'message': text[:2000]}})}\n\n"
                    return
                async for line in r.aiter_lines():
                    if not line.startswith("data:"):
                        continue
                    payload = line[5:].strip()
                    if payload == "[DONE]":
                        for idx, f in filters.items():
                            tail = f.flush()
                            if tail:
                                yield _delta_chunk(idx, tail)
                        yield "data: [DONE]\n\n"
                        break
                    chunk = json.loads(payload)
                    for choice in chunk.get("choices", []):
                        delta = choice.get("delta") or {}
                        delta.pop("reasoning_content", None)
                        delta.pop("reasoning", None)
                        if isinstance(delta.get("content"), str):
                            f = filters.setdefault(choice.get("index", 0), ThinkFilter(thinking))
                            delta["content"] = f.feed(delta["content"])
                    yield f"data: {json.dumps(chunk)}\n\n"
        except Exception as exc:
            _record_error(f"stream: {exc!r}")
            yield f"data: {json.dumps({'error': {'message': repr(exc)}})}\n\n"
        finally:
            STATS["last_latency_s"] = round(time.time() - t0, 3)

    return StreamingResponse(stream(), media_type="text/event-stream")


def _delta_chunk(index: int, text: str) -> str:
    chunk = {"object": "chat.completion.chunk",
             "choices": [{"index": index, "delta": {"content": text}, "finish_reason": None}]}
    return f"data: {json.dumps(chunk)}\n\n"


In [ ]:
%%writefile /content/llm_server/fallback_server.py
"""Fallback OpenAI-compatible server using plain `transformers` (text only).

Used automatically when vLLM cannot be installed, does not start in time, or
fails its validation. Implements POST /v1/chat/completions (temperature, top_p,
top_k, max_tokens, stop, stream, chat_template_kwargs) and GET /v1/models with
Bearer auth. One request is generated at a time (single GPU). Every failure is
returned as a JSON error with its message and logged with a traceback.

Configuration comes from environment variables:
  MODEL_ID, BACKEND_KEY, MAX_MODEL_LEN, ENABLE_THINKING, DEVICE, DTYPE
"""

from __future__ import annotations

import json
import os
import secrets
import threading
import time
import traceback
import uuid

import torch
from fastapi import FastAPI, Request
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import JSONResponse, StreamingResponse
from transformers import (
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoTokenizer,
    TextIteratorStreamer,
)

MODEL_ID = os.environ["MODEL_ID"]
API_KEY = os.environ.get("BACKEND_KEY", "")
MAX_MODEL_LEN = int(os.environ.get("MAX_MODEL_LEN", "8192"))
DEFAULT_THINKING = os.environ.get("ENABLE_THINKING", "false").lower() == "true"
DEVICE = os.environ.get("DEVICE", "cuda")
DTYPE = getattr(torch, os.environ.get("DTYPE", "float16"))  # T4 has no bf16


def _load():
    """Text-only language model first; the full vision-language class as fallback."""
    errors = []
    for cls in (AutoModelForCausalLM, AutoModelForImageTextToText):
        try:
            m, info = cls.from_pretrained(MODEL_ID, dtype=DTYPE, attn_implementation="sdpa",
                                          output_loading_info=True)
            missing = sorted(info.get("missing_keys", []))
            print(f"loaded {type(m).__name__} ({cls.__name__}); missing weights: "
                  f"{len(missing)} {missing[:5]}", flush=True)
            if len(missing) > 10:
                raise RuntimeError(f"{len(missing)} missing weights; refusing a partly "
                                   "random model")
            return m.to(DEVICE).eval()
        except Exception as exc:  # try the next class, report all if none works
            errors.append(f"{cls.__name__}: {type(exc).__name__}: {exc}")
            print(f"load with {cls.__name__} failed: {exc}", flush=True)
    raise RuntimeError("could not load the model: " + " | ".join(errors))


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = _load()
GEN_LOCK = threading.Lock()
CREATED = int(time.time())
print(f"ready: {MODEL_ID} on {DEVICE} ({DTYPE})", flush=True)

app = FastAPI(title="transformers fallback")


def _error(status: int, message: str) -> JSONResponse:
    return JSONResponse({"error": {"message": message, "type": "fallback_error"}},
                        status_code=status)


@app.middleware("http")
async def auth(request: Request, call_next):
    if API_KEY:
        header = request.headers.get("authorization", "")
        token = header[7:] if header.lower().startswith("bearer ") else ""
        if not (token and secrets.compare_digest(token, API_KEY)):
            return _error(401, "invalid or missing API key")
    return await call_next(request)


@app.get("/v1/models")
async def models():
    return {"object": "list", "data": [
        {"id": MODEL_ID, "object": "model", "created": CREATED, "owned_by": "fallback",
         "max_model_len": MAX_MODEL_LEN}]}


def _text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(p.get("text", "") for p in content if isinstance(p, dict))
    return "" if content is None else str(content)


def _prepare(body: dict):
    messages = [{"role": m["role"], "content": _text(m.get("content"))}
                for m in body.get("messages", [])]
    if not messages:
        raise ValueError("messages must not be empty")
    kwargs = dict(body.get("chat_template_kwargs") or {})
    kwargs.setdefault("enable_thinking", DEFAULT_THINKING)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                           **kwargs)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    n_prompt = int(inputs["input_ids"].shape[1])
    if n_prompt >= MAX_MODEL_LEN:
        raise ValueError(f"prompt has {n_prompt} tokens; MAX_MODEL_LEN is {MAX_MODEL_LEN}")
    max_new = int(body.get("max_tokens") or body.get("max_completion_tokens") or 512)
    max_new = max(1, min(max_new, MAX_MODEL_LEN - n_prompt))
    temperature = float(body.get("temperature", 0.7) or 0.0)
    gen = {"max_new_tokens": max_new, "do_sample": temperature > 0,
           "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
           # fp16 on a T4 can produce inf/nan logits; never crash sampling on them
           "remove_invalid_values": True}
    if temperature > 0:
        gen["temperature"] = temperature
        gen["top_p"] = float(body.get("top_p", 0.95))
        if body.get("top_k"):
            gen["top_k"] = int(body["top_k"])
    else:
        gen.update({"temperature": None, "top_p": None, "top_k": None})
    stop = body.get("stop")
    if stop:
        gen["stop_strings"] = [stop] if isinstance(stop, str) else list(stop)
        gen["tokenizer"] = tokenizer
    return inputs, n_prompt, gen


def _cut_at_stop(text: str, stop) -> str:
    for s in ([stop] if isinstance(stop, str) else (stop or [])):
        if s and s in text:
            text = text.split(s, 1)[0]
    return text


def _generate(inputs, gen):
    with GEN_LOCK, torch.inference_mode():
        return model.generate(**inputs, **gen)


@app.post("/v1/chat/completions")
async def chat(request: Request):
    rid = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    t0 = time.time()
    try:
        body = await request.json()
        if body.get("n", 1) != 1:
            return _error(400, "only n=1 is supported")
        inputs, n_prompt, gen = _prepare(body)
    except Exception as exc:
        print(f"{rid} bad request: {exc!r}", flush=True)
        return _error(400, f"{type(exc).__name__}: {exc}")

    if not body.get("stream"):
        try:
            out = await run_in_threadpool(_generate, inputs, gen)
        except Exception as exc:
            print(f"{rid} generation failed:\n{traceback.format_exc()}", flush=True)
            if isinstance(exc, torch.cuda.OutOfMemoryError):
                torch.cuda.empty_cache()
            return _error(500, f"generation failed: {type(exc).__name__}: {exc}")
        new_tokens = out[0, n_prompt:]
        text = _cut_at_stop(tokenizer.decode(new_tokens, skip_special_tokens=True),
                            body.get("stop"))
        n_out = int(new_tokens.shape[0])
        print(f"{rid} {n_prompt} prompt + {n_out} new tokens in {time.time() - t0:.1f}s",
              flush=True)
        finish = "length" if n_out >= gen["max_new_tokens"] else "stop"
        return {"id": rid, "object": "chat.completion", "created": int(time.time()),
                "model": MODEL_ID,
                "choices": [{"index": 0, "message": {"role": "assistant", "content": text},
                             "finish_reason": finish}],
                "usage": {"prompt_tokens": n_prompt, "completion_tokens": n_out,
                          "total_tokens": n_prompt + n_out}}

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    failure: list[str] = []

    def run():
        try:
            _generate(inputs, {**gen, "streamer": streamer})
        except Exception as exc:
            print(f"{rid} streaming generation failed:\n{traceback.format_exc()}", flush=True)
            failure.append(f"{type(exc).__name__}: {exc}")
            streamer.end()

    threading.Thread(target=run, daemon=True).start()

    def chunk(delta: dict, finish=None) -> str:
        data = {"id": rid, "object": "chat.completion.chunk", "created": int(time.time()),
                "model": MODEL_ID,
                "choices": [{"index": 0, "delta": delta, "finish_reason": finish}]}
        return f"data: {json.dumps(data)}\n\n"

    def events():
        yield chunk({"role": "assistant", "content": ""})
        for piece in streamer:
            if piece:
                yield chunk({"content": piece})
        if failure:
            yield f"data: {json.dumps({'error': {'message': failure[0]}})}\n\n"
        yield chunk({}, "stop")
        yield "data: [DONE]\n\n"

    return StreamingResponse(events(), media_type="text/event-stream")


In [ ]:
%%writefile /content/llm_server/llm_selftest.py
"""Validation of an OpenAI-compatible endpoint (used for the backend, the proxy and the
public URL). Never raises: every check records its status, timing and error body.

Checks
  plain     arithmetic answer with thinking off (must contain 42 when check_answers)
  json      a JSON-only answer that parses
  thinking  the same question with thinking on (must return 200 and an answer)
  stream    streamed answer ends with [DONE] and carries no error chunk
  long      a 400-token generation completes (catches crashes on longer outputs)
"""

from __future__ import annotations

import json
import re
import time

import httpx

QUESTION = "What is 17 + 25? Answer with the number only."


def strip_think(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[1]
    return text.replace("<think>", "").strip()


def call(base: str, key: str, body: dict, timeout: float = 900) -> tuple[int, object, float]:
    """POST /chat/completions; returns (status, parsed JSON or text, seconds)."""
    t0 = time.time()
    try:
        r = httpx.post(f"{base.rstrip('/')}/chat/completions", json=body,
                       headers={"Authorization": f"Bearer {key}"}, timeout=timeout)
    except httpx.HTTPError as exc:
        return 0, f"{type(exc).__name__}: {exc}", time.time() - t0
    try:
        data = r.json()
    except ValueError:
        data = r.text
    return r.status_code, data, time.time() - t0


def _content(data) -> tuple[str, str]:
    """(raw content, cleaned answer) from a chat completion."""
    try:
        msg = data["choices"][0]["message"]
    except (TypeError, KeyError, IndexError):
        return "", ""
    raw = msg.get("content") or msg.get("reasoning_content") or ""
    return raw, strip_think(raw)


def _err(data) -> str:
    if isinstance(data, dict) and "error" in data:
        e = data["error"]
        return e.get("message", str(e)) if isinstance(e, dict) else str(e)
    return str(data)[:800]


def validate(base: str, key: str, model: str, check_answers: bool = True,
             require_clean: bool = False) -> tuple[bool, list[dict]]:
    """Run every check; ``base`` ends with /v1. Returns (all required passed, rows)."""
    rows: list[dict] = []

    def body(content: str, thinking: bool, max_tokens: int, **extra) -> dict:
        return {"model": model, "temperature": 0, "max_tokens": max_tokens,
                "messages": [{"role": "user", "content": content}],
                "chat_template_kwargs": {"enable_thinking": thinking}, **extra}

    def add(name, ok, status, secs, detail, required=True):
        rows.append({"check": name, "ok": bool(ok), "status": status,
                     "seconds": round(secs, 1), "detail": str(detail)[:600],
                     "required": required})

    # plain answer, thinking off
    status, data, secs = call(base, key, body(QUESTION, False, 64))
    raw, ans = _content(data)
    ok = status == 200 and bool(ans) and (not check_answers or "42" in ans)
    if require_clean and "<think>" in raw:
        ok = False
    add("plain", ok, status, secs, repr(ans[:200]) if status == 200 else _err(data))

    # JSON answer
    status, data, secs = call(base, key, body(
        'Reply with JSON only, no prose: {"ok": true, "sum": <17 + 25>}', False, 80))
    raw, ans = _content(data)
    parsed = None
    if status == 200:
        m = re.search(r"\{.*\}", ans, re.S)
        try:
            parsed = json.loads(m.group(0)) if m else None
        except ValueError:
            parsed = None
    ok = status == 200 and (not check_answers or isinstance(parsed, dict))
    add("json", ok, status, secs, repr(ans[:200]) if status == 200 else _err(data))

    # thinking on
    status, data, secs = call(base, key, body(QUESTION, True, 1024))
    raw, ans = _content(data)
    ok = status == 200 and (not check_answers or bool(ans))
    if require_clean and ("<think>" in raw or "</think>" in raw):
        ok = False
    add("thinking", ok, status, secs, repr(ans[:200]) if status == 200 else _err(data))

    # streaming
    t0 = time.time()
    text, done, error = "", False, ""
    try:
        with httpx.stream("POST", f"{base.rstrip('/')}/chat/completions",
                          json=body("Count from 1 to 5.", False, 32, stream=True),
                          headers={"Authorization": f"Bearer {key}"}, timeout=900) as r:
            status = r.status_code
            for line in r.iter_lines():
                if not line.startswith("data:"):
                    continue
                payload = line[5:].strip()
                if payload == "[DONE]":
                    done = True
                    break
                chunk = json.loads(payload)
                if "error" in chunk:
                    error = _err(chunk)
                    break
                for ch in chunk.get("choices", []):
                    text += (ch.get("delta") or {}).get("content") or ""
            if status != 200:
                error = r.read().decode(errors="replace")[:600] if not error else error
    except Exception as exc:
        status, error = 0, f"{type(exc).__name__}: {exc}"
    add("stream", status == 200 and done and not error and bool(text.strip()), status,
        time.time() - t0, error or repr(text[:120]))

    # longer generation
    status, data, secs = call(base, key, body(
        "Write about 300 words on why hard drives fail.", False, 400, temperature=0.7))
    raw, ans = _content(data)
    tokens = (data.get("usage") or {}).get("completion_tokens", 0) if isinstance(data, dict) \
        else 0
    add("long", status == 200 and bool(ans), status, secs,
        f"{tokens} tokens, {tokens / secs:.1f} tok/s" if status == 200 and secs else _err(data))

    return all(r["ok"] for r in rows if r["required"]), rows


def report(label: str, rows: list[dict]) -> None:
    print(f"--- {label} ---")
    for r in rows:
        mark = "PASS" if r["ok"] else "FAIL"
        print(f"{mark}  {r['check']:9} HTTP {r['status']:<3} {r['seconds']:>6}s  {r['detail']}")


In [ ]:
import json, os, signal, subprocess, sys, time
import httpx
sys.path.insert(0, str(WORKDIR))
import importlib, llm_selftest
importlib.reload(llm_selftest)

def start_process(cmd, log_path, env=None):
    fh = open(log_path, "w")
    return subprocess.Popen(list(map(str, cmd)), stdout=fh, stderr=subprocess.STDOUT,
                            cwd=str(WORKDIR), env={**os.environ, **(env or {})},
                            start_new_session=True)

def stop_process(proc, name):
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=60)
    except Exception:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    print(f"stopped {name}")

def gpu_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip()
    try:
        return int(out.splitlines()[0])
    except (ValueError, IndexError):
        return -1

def wait_gpu_free(limit_mib=1500, timeout=120):
    t0 = time.time()
    while time.time() - t0 < timeout and gpu_used_mib() > limit_mib:
        time.sleep(3)
    print(f"GPU memory in use: {gpu_used_mib()} MiB")

def wait_ready(url, key, proc, timeout, name):
    t0, last = time.time(), 0
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            print(f"{name} exited with code {proc.returncode}")
            return False
        try:
            r = httpx.get(f"{url}/v1/models", headers={"Authorization": f"Bearer {key}"}, timeout=5)
            if r.status_code == 200:
                print(f"{name} ready after {time.time() - t0:.0f}s")
                return True
        except httpx.HTTPError:
            pass
        if time.time() - last > 30:
            print(f"  waiting for {name}... {time.time() - t0:.0f}s")
            last = time.time()
        time.sleep(5)
    print(f"{name} not ready after {timeout}s")
    return False

def validate(url, key, label, **kw):
    ok, rows = llm_selftest.validate(f"{url}/v1", key, MODEL_ID, **kw)
    llm_selftest.report(label, rows)
    return ok, rows

def stop_leftovers():
    """Free the ports and GPU if an earlier run or session left servers behind."""
    for name in ("proxy_proc", "backend_proc"):
        stop_process(globals().get(name), name)
    for pattern in ("vllm", "fallback_server:app", "proxy:app"):
        subprocess.run(["pkill", "-9", "-i", "-f", pattern])
    wait_gpu_free()

stop_leftovers()

BACKEND_URL = f"http://127.0.0.1:{BACKEND_PORT}"
backend_proc = None
STATE["backend"] = None  # choose again on every run

# ---------------------------------------------------------------- vLLM (primary)
if BACKEND in ("auto", "vllm") and VLLM_INSTALLED:
    vllm_cmd = [VLLM_ENV / "bin" / "vllm", "serve", MODEL_ID,
                "--host", "127.0.0.1", "--port", BACKEND_PORT,
                "--dtype", "float16", "--max-model-len", MAX_MODEL_LEN,
                "--gpu-memory-utilization", GPU_MEMORY_UTILIZATION, "--max-num-seqs", 16,
                "--api-key", BACKEND_KEY, "--served-model-name", MODEL_ID,
                "--language-model-only"]
    if VLLM_ENFORCE_EAGER:
        vllm_cmd.append("--enforce-eager")
    print("$", " ".join(map(str, vllm_cmd)).replace(BACKEND_KEY, "***"))
    backend_proc = start_process(vllm_cmd, LOGS["vllm"])
    if wait_ready(BACKEND_URL, BACKEND_KEY, backend_proc, STARTUP_TIMEOUT_S, "vLLM"):
        ok, _ = validate(BACKEND_URL, BACKEND_KEY, "vLLM backend validation")
        if ok:
            STATE["backend"] = "vllm"
        else:
            STATE["vllm_error"] = "validation failed"
    else:
        STATE["vllm_error"] = "did not start"
    if STATE["backend"] is None:
        print(f"\nvLLM rejected ({STATE['vllm_error']}). Last lines of vllm.log:\n" + log_tail(LOGS["vllm"], 30))
        stop_process(backend_proc, "vLLM")
        if BACKEND == "vllm":
            raise RuntimeError("vLLM failed and BACKEND='vllm' was forced; set BACKEND='auto' or 'transformers'.")
        wait_gpu_free()

# ------------------------------------------------------- transformers fallback
if STATE["backend"] is None:
    print("\nStarting the transformers fallback backend (first start downloads the model)...")
    backend_proc = start_process(
        [sys.executable, "-m", "uvicorn", "fallback_server:app", "--host", "127.0.0.1",
         "--port", BACKEND_PORT], LOGS["fallback"],
        env={"MODEL_ID": MODEL_ID, "BACKEND_KEY": BACKEND_KEY, "MAX_MODEL_LEN": str(MAX_MODEL_LEN),
             "ENABLE_THINKING": str(ENABLE_THINKING).lower(), "DEVICE": "cuda", "DTYPE": "float16"})
    started = wait_ready(BACKEND_URL, BACKEND_KEY, backend_proc, STARTUP_TIMEOUT_S, "fallback")
    ok = started and validate(BACKEND_URL, BACKEND_KEY, "fallback backend validation")[0]
    if not ok:
        STATE["fallback_error"] = "did not start" if not started else "validation failed"
        print("\nLast lines of fallback.log:\n" + log_tail(LOGS["fallback"], 40))
        raise RuntimeError(
            f"No working backend (vLLM: {STATE['vllm_error']}; fallback: {STATE['fallback_error']}). "
            "Read the validation table and logs above, or run the Troubleshooting cell.")
    STATE["backend"] = "transformers"

# ----------------------------------------------------------------------- proxy
proxy_proc = start_process(
    [sys.executable, "-m", "uvicorn", "proxy:app", "--host", "127.0.0.1", "--port", PORT],
    LOGS["proxy"],
    env={"LLM_API_KEY": LLM_API_KEY, "BACKEND_URL": BACKEND_URL, "BACKEND_KEY": BACKEND_KEY,
         "ENABLE_THINKING": str(ENABLE_THINKING).lower(), "MODEL_ID": MODEL_ID})
LOCAL_URL = f"http://127.0.0.1:{PORT}"
if not wait_ready(LOCAL_URL, LLM_API_KEY, proxy_proc, 120, "proxy"):
    print(log_tail(LOGS["proxy"]))
    raise RuntimeError("proxy failed to start")
print(f"\nServing {MODEL_ID} via {STATE['backend']} at {LOCAL_URL} (auth required)")

## 7 · Thinking control (validated through the proxy)
Qwen3.5 thinks by default. The proxy sends `enable_thinking=false` unless a client asks
otherwise, and **always** removes `<think>…</think>` text and `reasoning_content`. This cell runs
the same validation through the proxy and requires every answer to be free of reasoning text.

In [ ]:
ok, _ = validate(LOCAL_URL, LLM_API_KEY, "through the proxy (thinking stripped)", require_clean=True)
if not ok:
    print("\nproxy.log:\n" + log_tail(LOGS["proxy"], 30))
    raise RuntimeError("The proxy failed validation; see the table and log above.")
print("Thinking control: PASS (no reasoning text reaches clients)")

## 8 · Public tunnel
`cloudflared` quick tunnels need no account; `ngrok` needs `NGROK_TOKEN`. The public URL is
captured automatically and checked before use.

In [ ]:
import re, stat, urllib.request

tunnel_proc, PUBLIC_URL = None, None
if TUNNEL == "cloudflared":
    cf = WORKDIR / "cloudflared"
    if not cf.exists():
        url = (f"https://github.com/cloudflare/cloudflared/releases/download/"
               f"{VERSIONS['cloudflared']}/cloudflared-linux-amd64")
        urllib.request.urlretrieve(url, cf)
        cf.chmod(cf.stat().st_mode | stat.S_IEXEC)
    tunnel_proc = start_process([cf, "tunnel", "--no-autoupdate", "--url", LOCAL_URL], LOGS["tunnel"])
    t0 = time.time()
    while time.time() - t0 < 90 and PUBLIC_URL is None:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", LOGS["tunnel"].read_text(errors="replace"))
        PUBLIC_URL = m.group(0) if m else None
        time.sleep(1)
else:
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = NGROK_TOKEN
    PUBLIC_URL = ngrok.connect(PORT, "http").public_url.replace("http://", "https://")

if not PUBLIC_URL:
    print(log_tail(LOGS["tunnel"]))
    raise RuntimeError("Could not obtain a public tunnel URL")

# the new hostname can take a little while to resolve
t0, reachable = time.time(), False
while time.time() - t0 < 180 and not reachable:
    try:
        reachable = httpx.get(f"{PUBLIC_URL}/v1/models", headers={"Authorization": f"Bearer {LLM_API_KEY}"},
                              timeout=10).status_code == 200
    except httpx.HTTPError:
        pass
    if not reachable:
        time.sleep(3)
print("Public URL:", PUBLIC_URL, "| reachable:", reachable)
if not reachable:
    print("The tunnel is up but not answering yet; step 10 retries. Tunnel log:\n" + log_tail(LOGS["tunnel"], 15))

## 9 · Connection block
Paste this into `server/.env` (or the web app's **Settings** page).

In [ ]:
LLM_BASE_URL = f"{PUBLIC_URL}/v1"
print("=" * 64)
print(f"LLM_BASE_URL={LLM_BASE_URL}")
print(f"LLM_API_KEY={LLM_API_KEY}")
print(f"LLM_MODEL={MODEL_ID}")
print("=" * 64)
print(f"backend: {STATE['backend']} | thinking default: {ENABLE_THINKING}")

## 10 · Self-test through the public URL
This calls the endpoint the same way the project's backend does:
- (a) the full validation through the tunnel;
- (b) the `openai` client with a JSON-schema answer parsed by Pydantic;
- (c) latency and tokens per second;
- (d) a request without the key must be rejected.

Each line prints PASS or FAIL with the reason.

In [ ]:
from openai import OpenAI, AuthenticationError
from pydantic import BaseModel

results = {}
ok, rows = validate(PUBLIC_URL, LLM_API_KEY, "(a) public URL validation", require_clean=True)
results["a_public_validation"] = ok

class DriveAssessment(BaseModel):
    risk_level: str
    reasons: list[str]

client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, timeout=600, max_retries=1)
EXTRA = {"chat_template_kwargs": {"enable_thinking": False}}
schema = json.dumps(DriveAssessment.model_json_schema())
try:
    r = client.chat.completions.create(model=MODEL_ID, temperature=0, max_tokens=300, extra_body=EXTRA,
        messages=[{"role": "user", "content": "A hard drive's reallocated sector count rose from 0 to 48 in "
                   f"10 days. Return ONLY a JSON object matching this JSON schema, no prose: {schema}"}])
    raw = (r.choices[0].message.content or "").strip()
    m = re.search(r"\{.*\}", raw, re.S)
    print("(b)", DriveAssessment.model_validate_json(m.group(0) if m else raw))
    results["b_json_pydantic"] = True
except Exception as exc:
    results["b_json_pydantic"] = False
    print("(b) FAILED:", type(exc).__name__, str(exc)[:500])

try:
    t0 = time.time()
    r = client.chat.completions.create(model=MODEL_ID, temperature=0.7, max_tokens=200, extra_body=EXTRA,
        messages=[{"role": "user", "content": "Write a 150-word paragraph about hard drive reliability."}])
    dt, toks = time.time() - t0, (r.usage.completion_tokens if r.usage else 0)
    print(f"(c) latency {dt:.2f}s | {toks} tokens | {toks / dt:.1f} tokens/s")
    results["c_latency"] = toks > 0
except Exception as exc:
    results["c_latency"] = False
    print("(c) FAILED:", type(exc).__name__, str(exc)[:500])

try:
    OpenAI(base_url=LLM_BASE_URL, api_key="wrong-key", max_retries=0).models.list()
    results["d_auth_enforced"] = False
except AuthenticationError:
    results["d_auth_enforced"] = True
except Exception as exc:
    results["d_auth_enforced"] = False
    print("(d) unexpected error:", exc)
try:
    results["d_auth_enforced"] &= httpx.get(f"{LLM_BASE_URL}/models", timeout=30).status_code == 401
except httpx.HTTPError:
    results["d_auth_enforced"] = False

print()
for name, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
print("ALL PASS" if all(results.values()) else "SOME TESTS FAILED: see the reasons above, or run Troubleshooting")

## 11 · Keep-alive monitor
Prints health every 60 s: GPU memory, requests, errors, last error and process status. Leave it
running to keep the session active; **interrupt** it (■) before running Shutdown or
Troubleshooting.

In [ ]:
def monitor(interval=60):
    print("Monitoring; interrupt this cell to stop.")
    while True:
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
        try:
            s = httpx.get(f"{LOCAL_URL}/stats", headers={"Authorization": f"Bearer {LLM_API_KEY}"},
                          timeout=10).json()
        except Exception as exc:
            s = {"error": repr(exc)}
        alive = {"backend": backend_proc.poll() is None, "proxy": proxy_proc.poll() is None,
                 "tunnel": tunnel_proc.poll() is None if tunnel_proc else True}
        print(f"[{time.strftime('%H:%M:%S')}] GPU {gpu} | requests={s.get('requests')} "
              f"errors={s.get('errors')} last_latency={s.get('last_latency_s')}s "
              f"last_error={s.get('last_error')} | alive={alive} | {LLM_BASE_URL}")
        if not all(alive.values()):
            print("A process stopped; run the Troubleshooting cell.")
        time.sleep(interval)

try:
    monitor()
except KeyboardInterrupt:
    print("monitor stopped")

## 12 · Shutdown
Stops the tunnel, proxy and backend cleanly. Run it manually.

In [ ]:
if TUNNEL == "ngrok":
    try:
        from pyngrok import ngrok
        ngrok.kill()
    except Exception as exc:
        print("ngrok:", exc)
stop_process(globals().get("tunnel_proc"), "tunnel")
stop_process(globals().get("proxy_proc"), "proxy")
stop_process(globals().get("backend_proc"), STATE.get("backend") or "backend")
print("All stopped.")

## Troubleshooting (run manually)
Prints the state, process status, GPU memory, one request through each layer with its full error
text, and the end of every log. Copy the output when asking for help (the key is masked).

| Symptom | Fix |
|---|---|
| vLLM rejected, fallback works | Nothing to do: the fallback is slower but complete. To skip vLLM next time, set `BACKEND = "transformers"`. |
| `CUDA out of memory` | Lower `MAX_MODEL_LEN` (e.g. 4096) or `GPU_MEMORY_UTILIZATION` (e.g. 0.8), then Run all. |
| 401 | Wrong key: copy `LLM_API_KEY` from step 9 again. |
| 502 `backend unavailable` | The backend process died: see its log below, then Run all. |
| Tunnel not reachable | Wait a minute, or switch `TUNNEL` to `ngrok`. |

In [ ]:
def mask(text):
    text = str(text)
    for secret in (globals().get("LLM_API_KEY"), globals().get("BACKEND_KEY")):
        if secret:
            text = text.replace(secret, "***")
    return text

print("state:", STATE)
for name in ("backend_proc", "proxy_proc", "tunnel_proc"):
    p = globals().get(name)
    print(f"{name}: {'not started' if p is None else ('running' if p.poll() is None else f'exited {p.returncode}')}")
print("GPU memory in use (MiB):", gpu_used_mib() if "gpu_used_mib" in globals() else "n/a")
probe = {"model": MODEL_ID, "max_tokens": 32, "temperature": 0,
         "messages": [{"role": "user", "content": "What is 17 + 25? Answer with the number only."}]}
for label, url, key in (("backend", globals().get("BACKEND_URL"), globals().get("BACKEND_KEY")),
                        ("proxy", globals().get("LOCAL_URL"), globals().get("LLM_API_KEY")),
                        ("public", globals().get("PUBLIC_URL"), globals().get("LLM_API_KEY"))):
    if not url:
        print(f"\n{label}: not available")
        continue
    body = {**probe, "chat_template_kwargs": {"enable_thinking": False}}
    try:
        r = httpx.post(f"{url}/v1/chat/completions", json=body,
                       headers={"Authorization": f"Bearer {key}"}, timeout=300)
        print(f"\n{label}: HTTP {r.status_code}\n{mask(r.text[:1500])}")
    except Exception as exc:
        print(f"\n{label}: {type(exc).__name__}: {exc}")
for name, path in LOGS.items():
    print(f"\n----- {name}.log (last 40 lines) -----\n{mask(log_tail(path, 40))}")